# PoliMillionaire — Math Strategies (Colab-ready)

This notebook documents **every strategy** we built and tested for the **PoliMillionaire** NLP project (Politecnico di Milano, 2025/26).

The game is an online multiple-choice quiz played via API. We focus exclusively on the **Maths category** (comp_id = 3) with a **30-second time limit** per question.

To compare strategies objectively, we collected a **fixed set of ~47 real PoliMillionaire Maths questions** (with known correct answers) by playing actual games and recording every question seen. This benchmark (`math_questions.json`) allows us to run every strategy on exactly the same questions and measure accuracy under identical conditions — something impossible with the live game, which returns different questions each round.

---

## Strategy overview

| # | Name | Idea | Benchmark acc | Avg time |
|---|------|------|--------------|----------|
| 1 | Zero-shot baseline | Plain MCQ prompt | 59.6% | 0.4 s |
| 2 | Chain-of-Thought | Brief reasoning + Answer: X | 70.2% | 3.2 s |
| 3 | Semantic few-shot | MMLU examples retrieved by embedding | 55.3% | 0.8 s |
| 4 | Hybrid-Math v1 (classifier) | LLM decides: compute or reason | 63.8% | 7.9 s |
| 5 | Hybrid-Math Llama | Always try Python code first, then reason | 83.0% | 21.5 s |
| 6 | DeepSeek-R1 7B | Chain-of-thought reasoning model | 68.1% | 45.3 s ⚠️ |
| 7 | DeepSeek-R1 1.5B | Fast but weak version of DeepSeek | 51.1% | 7.4 s |
| 8 | Hybrid-Math DeepSeek | Code (Llama) + reasoning (DeepSeek 1.5b) | 59.6% | 11.0 s |
| 9 | Hybrid-Math Qwen | Same as #5 but with Qwen2.5-7B | 81% | 9.6 |
| 10 | Prompt Engineering | Find the best prompt style empirically | 59.6% | 25.7 s |
| 11 | Multi-Model Benchmark | Compare qwen / mistral / llama side by side | 70.2% | 1.0 s |
| 12 | Parallel Ensemble | 3 models vote in parallel, confidence-weighted | 70.2% | 16.0 s |
| 13 | Fine-tuned Qwen2.5 | LoRA fine-tuned on MathQA + MMLU-Pro | 76.6% | ~5 s |

---

## How to run on Google Colab (free T4)

1. Open the **Colab setup** cell and run it first — it installs packages and starts Ollama.
2. Add `POLI_USER` and `POLI_PASS` in the **Secrets panel** (🔑 key icon, left sidebar). Enable **Notebook access** for both.
3. Upload `math_questions.json` when prompted.
4. Run the **Shared setup** cell (login + helpers).
5. Run the **Load benchmark** cell.
6. **Run ONE strategy cell at a time.** Each cell is self-contained and pulls its model automatically.
7. After testing all strategies, run **Final comparison**.

> **Memory note (T4 = 15 GB VRAM):** A single 7–8 B model at Q4 uses ~4–5 GB.  
> The parallel ensemble (Strategy 12) loads 3 models = ~12 GB — within limits.  
> If you get OOM, remove one model from `_ENS_MODELS`.


## Step 1 — Colab setup *(run only on Google Colab)*

Run this cell first. It installs all packages, starts the Ollama server with multi-model support, and prompts you to upload `math_questions.json`.

If you are running **locally** (VS Code / Jupyter), skip this cell.

In [16]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import subprocess, time

    # ── Install Python packages ──────────────────────────────────────────
    os.system('pip install -q ollama python-dotenv millionaire_client '
              'sentence-transformers datasets requests')

    # ── Install zstd (required by ollama install script) ─────────────────
    os.system('apt-get install -y zstd -qq')

    # ── Install Ollama ───────────────────────────────────────────────────
    _OLLAMA_BIN = '/usr/local/bin/ollama'
    if not os.path.exists(_OLLAMA_BIN):
        print('Installing Ollama...')
        os.system('curl -fsSL https://ollama.com/install.sh | sh')
        if not os.path.exists(_OLLAMA_BIN):
            raise RuntimeError(
                'Ollama installation failed.\n'
                'Prova a runnare manualmente in una nuova cella:\n'
                '  !apt-get install -y zstd && curl -fsSL https://ollama.com/install.sh | sh'
            )
        print('Ollama installed.')
    else:
        print('Ollama already installed.')

    # ── Start Ollama server ───────────────────────────────────────────────
    os.system('pkill -f "ollama serve" 2>/dev/null || true')
    time.sleep(1)
    os.system(
        'OLLAMA_MAX_LOADED_MODELS=3 OLLAMA_NUM_PARALLEL=3 '
        'nohup ollama serve > /tmp/ollama.log 2>&1 &'
    )
    time.sleep(5)

    if os.system('curl -sf http://localhost:11434 > /dev/null') != 0:
        print('WARNING: Ollama server not responding. Log:')
        os.system('tail -20 /tmp/ollama.log')
    else:
        print('Ollama server started (OLLAMA_MAX_LOADED_MODELS=3, OLLAMA_NUM_PARALLEL=3).')

    # ── Upload math_questions.json ────────────────────────────────────────
    if not os.path.exists('math_questions.json'):
        from google.colab import files
        print('Please upload math_questions.json:')
        uploaded = files.upload()
        if 'math_questions.json' not in uploaded:
            print('WARNING: math_questions.json not uploaded — benchmark mode will not work.')
    else:
        print('math_questions.json already present.')

    print('\nColab setup complete.')
    print('Next: add POLI_USER and POLI_PASS in the Secrets panel (🔑 left sidebar).')

else:
    print('Local environment — skipping Colab setup.')
    print('For the parallel ensemble (Strategy 12) make sure Ollama is running with:')
    print('  OLLAMA_MAX_LOADED_MODELS=3 OLLAMA_NUM_PARALLEL=3 ollama serve')

Ollama already installed.
Ollama server started (OLLAMA_MAX_LOADED_MODELS=3, OLLAMA_NUM_PARALLEL=3).
Please upload math_questions.json:


Saving math_questions.json to math_questions.json

Colab setup complete.
Next: add POLI_USER and POLI_PASS in the Secrets panel (🔑 left sidebar).


In [6]:
!curl -fsSL https://ollama.com/install.sh | sh
!apt-get install -y zstd && curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
ERROR: This version requires zstd for extraction. Please install zstd and try again:
  - Debian/Ubuntu: sudo apt-get install zstd
  - RHEL/CentOS/Fedora: sudo dnf install zstd
  - Arch: sudo pacman -S zstd
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (8,466 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ..

## Step 2 — Shared setup

Run this cell once per session. It handles:
- Imports and login to the PoliMillionaire API
- The `_llm()` wrapper around Ollama
- The `run_strategy()` harness (live game and offline benchmark)
- Plotting helpers
- `_colab_pull()` — auto-downloads any missing Ollama model before each strategy


In [17]:
# ════════════════════════════════════════════════════════════════════════
#  SHARED SETUP
# ════════════════════════════════════════════════════════════════════════
import sys, os, importlib, subprocess as _sp

IN_COLAB = 'google.colab' in sys.modules

# ── Mount Drive and install millionaire_client ───────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    _MC_SRC = '/content/drive/MyDrive/NLP_project/millionaire_client'
    _MC_DST = '/content/millionaire_client'

    if not os.path.isdir(_MC_SRC):
        raise RuntimeError(f'millionaire_client not found at {_MC_SRC} — check your Drive path.')

    # Copy locally (Drive FUSE is slow and pip -e can fail on it)
    os.system(f'cp -r "{_MC_SRC}" "{_MC_DST}"')

    # Try editable install, then normal install, then just add to sys.path
    ret = os.system(f'pip install -q -e "{_MC_DST}" 2>/dev/null')
    if ret != 0:
        ret = os.system(f'pip install -q "{_MC_DST}" 2>/dev/null')
    if ret != 0:
        # No setup.py at all — just put the folder on the path
        if _MC_DST not in sys.path:
            sys.path.insert(0, '/content')
        print('millionaire_client added to sys.path directly.')
    else:
        print('millionaire_client installed.')
# ── Install any other missing packages ───────────────────────────────────
for _pkg, _mod in [('ollama', 'ollama'), ('python-dotenv', 'dotenv'),
                   ('sentence-transformers', 'sentence_transformers'),
                   ('datasets', 'datasets')]:
    if importlib.util.find_spec(_mod) is None:
        _sp.run(['pip', 'install', '-q', _pkg], check=True)

import os, sys, re, time, random, json
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import ollama

sys.path.insert(0, os.path.abspath('./'))
from millionaire_client import MillionaireClient
try:
    import dotenv; dotenv.load_dotenv()
except ImportError:
    pass

# ── Login ────────────────────────────────────────────────────────────────
client   = MillionaireClient('http://131.175.15.22:51111/')

if IN_COLAB:
    from google.colab import userdata as _ud
    _user = _ud.get('POLI_USER')
    _pass = _ud.get('POLI_PASS')
else:
    _user = os.getenv('POLI_USER') or os.getenv('USER')
    _pass = os.getenv('POLI_PASS') or os.getenv('PASSWORD')

user = client.login(_user, _pass)
print(f'Login OK. Welcome, {user.username}!')

competitions    = client.competitions.list_all()
COMPETITION_IDS = [c.id for c in competitions]
COMP_NAME       = {c.id: c.name for c in competitions}

# ── Constants ────────────────────────────────────────────────────────────
LABELS             = ['A', 'B', 'C', 'D']
N_GAMES            = 5
QUESTIONS_PER_GAME = 15

LLAMA   = 'llama3.1:8b'
MISTRAL = 'mistral:7b'
QWEN    = 'qwen2.5:7b'

VERBOSE    = globals().get('VERBOSE',    False)
SHOW_QA    = globals().get('SHOW_QA',    True)
SAVE_PLOTS = globals().get('SAVE_PLOTS', False)
PLOTS_DIR  = globals().get('PLOTS_DIR',  'plots')

# ── LLM wrapper ──────────────────────────────────────────────────────────
def _llm(prompt, max_tokens=200, model=None, temperature=0.0):
    resp = ollama.chat(
        model=model or LLAMA,
        messages=[{'role': 'user', 'content': prompt}],
        options={'num_predict': max_tokens, 'temperature': temperature},
    )['message']['content']
    if VERBOSE:
        print(f'\n  +-- [{model or LLAMA}]')
        for line in resp.splitlines(): print(f'  | {line}')
        print('  +--')
    return resp

# ── Harness ──────────────────────────────────────────────────────────────
RESULTS    = {}
STRATEGIES = {}

def _show_question(question):
    if SHOW_QA:
        print(f'\nQ: {question.text}')
        for i, o in enumerate(question.options):
            print(f'   {LABELS[i]}) {o.text}')

def _show_answer(question, chosen_id, is_ok=None, dt=None):
    if not SHOW_QA: return
    idx  = next(i for i, o in enumerate(question.options) if o.id == chosen_id)
    mark = '' if is_ok is None else ('  [CORRECT]' if is_ok else '  [WRONG]')
    secs = '' if dt is None else f'  ({dt:.1f}s)'
    print(f'-> {LABELS[idx]}) {question.options[idx].text}{mark}{secs}')

def play_game(competition_id, strategy_fn, game_index, after_answer=None, before_game=None):
    if before_game: before_game()
    game, correct, times = client.game.start(competition_id), 0, []
    while game.in_progress:
        q = game.current_question
        if not q: break
        _show_question(q)
        t0        = time.time()
        chosen_id = strategy_fn(q)
        dt        = time.time() - t0
        result    = game.answer(chosen_id)
        times.append(dt)
        if result.correct: correct += 1
        _show_answer(q, chosen_id, result.correct, dt)
        if after_answer: after_answer(q, next(o.text for o in q.options if o.id == chosen_id), result)
        if result.game_over: break
    acc  = correct / QUESTIONS_PER_GAME
    icon = 'OK ' if game.earned_amount > 0 else 'KO '
    print(f'    [{icon}] Game {game_index} | Level {game.current_level} | '
          f'Earned: {game.earned_amount:,.0f} | Acc: {acc:.0%}')
    return {'game': game_index, 'correct': correct, 'accuracy': round(acc, 4),
            'earned': game.earned_amount, 'reached_level': game.current_level,
            'response_time_avg': round(sum(times)/len(times), 3) if times else 0}

# ── Offline benchmark ────────────────────────────────────────────────────
class _Opt:
    def __init__(self, oid, text): self.id, self.text = oid, text
class _Q:
    def __init__(self, text, options, correct_id, level=None):
        self.text, self.options, self.correct_id, self.level = text, options, correct_id, level
class _FakeResult:
    def __init__(self, correct): self.correct, self.game_over = correct, False

BENCHMARK = None

def load_benchmark(path='math_questions.json', n=None, shuffle=True, seed=0):
    global BENCHMARK
    qs = []
    for it in json.load(open(path)):
        options    = [_Opt(i, it['options'][L]) for i, L in enumerate(LABELS)]
        correct_id = LABELS.index(it['correct_label'])
        qs.append(_Q(it['question'], options, correct_id, level=it.get('level')))
    if shuffle: random.Random(seed).shuffle(qs)
    if n:       qs = qs[:n]
    BENCHMARK = qs
    print(f'Benchmark loaded: {len(BENCHMARK)} questions -> OFFLINE mode ON.')
    return BENCHMARK

def use_live_game():
    global BENCHMARK
    BENCHMARK = None
    print('OFFLINE mode OFF -> run_strategy() plays the live game.')

def _run_offline(name, fn, after_answer=None, before_game=None):
    RESULTS.setdefault(name, {})
    n = len(BENCHMARK)
    print(f'\n=== {name} | OFFLINE on {n} questions ===')
    if before_game: before_game()
    correct, times = 0, []
    for q in BENCHMARK:
        _show_question(q)
        t0        = time.time()
        chosen_id = fn(q)
        dt        = time.time() - t0
        times.append(dt)
        is_ok  = (chosen_id == q.correct_id)
        correct += int(is_ok)
        _show_answer(q, chosen_id, is_ok, dt)
        if after_answer:
            after_answer(q, next(o.text for o in q.options if o.id == chosen_id), _FakeResult(is_ok))
    acc   = correct / n if n else 0
    avg_t = sum(times) / len(times) if times else 0
    print(f'    {correct}/{n} correct | Acc: {acc:.0%} | avg {avg_t:.1f}s/question')
    RESULTS[name]['Maths'] = [{'game': 1, 'label': f'All ({n} q)', 'correct': correct,
                               'accuracy': round(acc, 4), 'n': n, 'earned': 0,
                               'reached_level': 0, 'response_time_avg': round(avg_t, 3)}]
    return RESULTS[name]

def run_strategy(name, fn, comp_ids=None, after_answer=None, before_game=None):
    STRATEGIES[name] = dict(fn=fn, comp_ids=comp_ids,
                            after_answer=after_answer, before_game=before_game)
    if BENCHMARK is not None:
        return _run_offline(name, fn, after_answer, before_game)
    target_ids = comp_ids if comp_ids is not None else COMPETITION_IDS
    RESULTS.setdefault(name, {})
    print(f'\n=== {name} | {len(target_ids)} comp x {N_GAMES} games (LIVE) ===')
    for comp_id in target_ids:
        comp_name = COMP_NAME[comp_id]
        print(f'  --- {comp_name} ---')
        RESULTS[name][comp_name] = [play_game(comp_id, fn, i, after_answer, before_game)
                                    for i in range(1, N_GAMES + 1)]
    return RESULTS[name]

# ── Plotting ─────────────────────────────────────────────────────────────
def plot_comparison(comp_name='Maths'):
    names = [n for n in RESULTS if comp_name in RESULTS[n]]
    if not names:
        print(f'No results for competition "{comp_name}".'); return
    means = [np.mean([g['accuracy'] for g in RESULTS[n][comp_name]]) for n in names]
    times = [np.mean([g['response_time_avg'] for g in RESULTS[n][comp_name]]) for n in names]
    xs    = np.arange(len(names))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.bar(xs, means, color='steelblue')
    ax1.set_ylabel('Mean accuracy'); ax1.set_ylim(0, 1)
    ax1.set_title(f'Accuracy — {comp_name}'); ax1.grid(axis='y', alpha=0.3)
    for i, m in enumerate(means): ax1.text(i, m+0.01, f'{m:.0%}', ha='center', fontsize=8)
    ax2.bar(xs, times, color='indianred')
    ax2.set_ylabel('Avg response time (s)')
    ax2.set_title(f'Response time — {comp_name}'); ax2.grid(axis='y', alpha=0.3)
    ax2.axhline(30, color='black', ls='--', lw=1)
    ax2.text(len(names)-0.5, 31, '30 s limit', ha='right', fontsize=8)
    for i, t in enumerate(times): ax2.text(i, t+max(times)*0.01+0.1, f'{t:.1f}s', ha='center', fontsize=8)
    for ax in (ax1, ax2):
        ax.set_xticks(xs); ax.set_xticklabels(names, rotation=25, ha='right')
    plt.tight_layout(); plt.show()

def plot_strategy(name):
    data = RESULTS.get(name)
    if not data:
        print(f'No results for "{name}".'); return
    comps  = list(data.keys())
    fig, ax = plt.subplots(figsize=(8, 4))
    width   = 0.8 / max(len(comps), 1)
    for k, comp in enumerate(comps):
        accs = [g['accuracy'] for g in data[comp]]
        ax.bar(np.arange(len(accs)) + k * width, accs, width, label=comp)
    ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1)
    ax.set_title(f'"{name}" — accuracy per game')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

# ── Colab pull helper ────────────────────────────────────────────────────
def _colab_pull(*models, extra_pip=None):
    if extra_pip:
        os.system(f'pip install -q {extra_pip}')
    _resp = ollama.list()
    _raw  = _resp.get('models', []) if isinstance(_resp, dict) else getattr(_resp, 'models', [])
    existing = {(m['model'] if isinstance(m, dict) else m.model) for m in _raw}
    for m in models:
        if m in existing:
            print(f'[ollama] {m} already present, skipping pull.'); continue
        print(f'[ollama] pulling {m} ...')
        for chunk in ollama.pull(m, stream=True):
            status = chunk.get('status', '') if isinstance(chunk, dict) else getattr(chunk, 'status', '')
            total  = (chunk.get('total', 0) if isinstance(chunk, dict) else getattr(chunk, 'total', 0)) or 0
            done   = (chunk.get('completed') or 0) if isinstance(chunk, dict) else (getattr(chunk, 'completed', 0) or 0)
            if total: print(f'\r  {status}: {done/total*100:.1f}%', end='', flush=True)
            elif status: print(f'  {status}', flush=True)
        print(f'\n[ollama] {m} ready.')

print('Setup complete. Run the strategy cells below (each is independent).')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
millionaire_client added to sys.path directly.
Login OK. Welcome, Alisss!
Setup complete. Run the strategy cells below (each is independent).


## Step 3 — Load fixed benchmark

Enables **offline mode**: every strategy runs on the same 47 fixed Maths questions with known answers, making results directly comparable across strategies.

Call `use_live_game()` to switch back to the real game.

In [18]:
load_benchmark()          # reads math_questions.json from the current directory
# load_benchmark(n=20)   # optional: run only the first 20 questions (faster)
# use_live_game()        # switch back to the live game


Benchmark loaded: 47 questions -> OFFLINE mode ON.


---
## Strategy 1 — Zero-shot baseline

**Starting point.** Before building anything, we need to know how well a general-purpose LLM performs with zero guidance. We send the question and options as-is and ask for just the letter.

**What we expected.** A bare LLM should do better than random (25%) because it has been pre-trained on large amounts of text including math. But without explicit reasoning we expect it to pattern-match rather than calculate.

**What we found.** 59.6% accuracy, 0.4 s/question. Correct on easy definitional questions, but consistently wrong on anything requiring actual arithmetic. The model often picks the "most familiar-looking" number rather than computing the answer. This establishes the **lower bound** for our iterative improvement.

**Why it matters.** Every subsequent strategy is measured against this baseline. A technique that does not beat 59.6% is actively harmful.


In [19]:
# ── STRATEGY 1 — Zero-shot baseline ─────────────────────────────────────
_colab_pull('llama3.1:8b')
import re, random

def zero_shot(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    raw  = _llm(
        f'Answer this multiple choice question. Reply with ONLY the letter (A, B, C or D).\n\n'
        f'Question: {question.text}\n\n{opts}\n\nAnswer:',
        max_tokens=10,
    )
    for ch in reversed(raw.strip().upper()):
        if ch in LABELS:
            return question.options[LABELS.index(ch)].id
    return random.choice(question.options).id

run_strategy('zero-shot', zero_shot, comp_ids=[3])


[ollama] pulling llama3.1:8b ...
  pulling manifest
  pulling 455f34728c9b: 100.0%  verifying sha256 digest
  writing manifest
  success

[ollama] llama3.1:8b ready.

=== zero-shot | OFFLINE on 47 questions ===

Q: Compute the product in the given ring. (20)(-8) in Z_26
   A) 11
   B) 22
   C) 1
   D) 0
-> C) 1  [WRONG]  (117.6s)

Q: A company with offices in five different countries is interested in estimating the proportion of its employees in favor of banning smoking on the office premises. It is known that the views of people from different countries on issues like smoking tend to vary due to the influence of different local social structures. Which of the following is an appropriate sampling technique to use in this situation, and why?
   A) Simple random sample, because this is the simplest sampling method
   B) Stratified sample, because the population is divided into five strata—namely, five offices in five different countries
   C) Cluster sample, because the population is div

{'Maths': [{'game': 1,
   'label': 'All (47 q)',
   'correct': 28,
   'accuracy': 0.5957,
   'n': 47,
   'earned': 0,
   'reached_level': 0,
   'response_time_avg': 2.862}]}

## Strategy 2 — Chain-of-Thought (CoT)

**What changed from S1.** We add two words of instruction: "Reason briefly". This forces the model to generate intermediate steps before committing to a letter.

**Why CoT works (in general).** The "chain-of-thought" prompting technique (Wei et al., 2022) shows that requiring explicit intermediate reasoning dramatically improves performance on multi-step problems — because the model conditions its final answer on the reasoning trace rather than generating the letter from scratch.

**Key engineering detail.** The parser now searches for `Answer: X` on the last line rather than scanning the raw output for any letter. This eliminates false positives where the model mentions a letter mid-reasoning ("if we pick A we get...") and then gives the wrong final answer.

**What we found.** 70.2% accuracy, 3.2 s/question. +10.6 percentage points over zero-shot. Significant improvement on conceptual statistics questions. Still fails on questions requiring precise arithmetic — the model reasons qualitatively but does not actually compute.

**Key insight.** Words are not numbers. Natural language reasoning can describe *how* to solve a problem without actually solving it. This motivates a fundamentally different approach: execute real Python code.


In [ ]:
# ── STRATEGY 2 — Chain-of-Thought ────────────────────────────────────────
_colab_pull('llama3.1:8b')
import re, random

def _cot_parse(raw, question):
    upper = raw.strip().upper()
    m = re.search(r'ANSWER\s*[:=]\s*([A-D])', upper)
    if m: ch = m.group(1)
    else:
        m  = re.search(r'\b([A-D])\b\s*$', upper)
        ch = m.group(1) if m else None
    if ch is None:
        for c in reversed(upper):
            if c in LABELS: ch = c; break
    if ch is None: return random.choice(question.options).id
    return question.options[LABELS.index(ch)].id

def cot_fixed(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    raw  = _llm(
        f'Answer the multiple choice question below.\n'
        f'Reason briefly (2-3 sentences), then on the last line write exactly: Answer: X\n'
        f'(X must be A, B, C, or D)\n\nQuestion: {question.text}\n\n{opts}\n',
        max_tokens=300,
    )
    if SHOW_QA:
        reasoning = re.sub(r'(?is)\n?\s*answer\s*[:=].*$', '', raw).strip()
        if reasoning:
            for line in reasoning.splitlines():
                if line.strip(): print(f'   {line.strip()}')
    return _cot_parse(raw, question)

run_strategy('cot-fixed', cot_fixed, comp_ids=[3])


## Strategy 3 — Semantic Few-shot (MMLU)

**Hypothesis.** If we show the model examples of similar questions with correct answers, it can pattern-match on the question *type* and give better answers. We retrieve the 3 most semantically similar questions from MMLU using sentence embeddings (`all-MiniLM-L6-v2`).

**Why it should help.** In-context examples reduce ambiguity: the model sees what the expected output format looks like, and analogous examples can trigger the right reasoning pathway.

**Why it underperformed (55.3%).** Two problems:
1. **Domain mismatch.** MMLU covers many subjects. Even "mathematically similar" questions retrieved by embedding similarity are rarely close enough in *style* to PoliMillionaire's concise Italian-university-MCQ format.
2. **Embedding shallowness.** `all-MiniLM-L6-v2` matches surface vocabulary, not mathematical structure. A statistics question about variance and a calculus question about integrals both use formal language and get mixed up.

**What we learned.** Retrieval-augmented few-shot is not a free win. The quality of the example bank matters more than having examples at all. Rather than continuing down this path, we pivot: instead of better examples, we need a tool that can *actually compute* the answer.


In [ ]:
# ── STRATEGY 3 — Semantic few-shot with MMLU ─────────────────────────────
_colab_pull('llama3.1:8b', extra_pip='sentence-transformers datasets')
import re, random
import numpy as np
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

_MMLU_SUBJECTS = [
    'high_school_mathematics', 'high_school_statistics', 'college_mathematics',
    'high_school_physics', 'high_school_chemistry', 'high_school_biology',
    'high_school_world_history', 'high_school_european_history',
    'philosophy', 'logical_fallacies', 'miscellaneous',
]

print('Loading MMLU examples and building embedding bank...')
_fs_embedder = SentenceTransformer('all-MiniLM-L6-v2')
_fs_pairs, _fs_embs = [], None
for subj in _MMLU_SUBJECTS:
    try:
        ds = load_dataset('cais/mmlu', subj, split='test')
        for row in ds.select(range(min(25, len(ds)))):
            idx = row['answer']
            _fs_pairs.append((row['question'], f"{LABELS[idx]}) {row['choices'][idx]}"))
    except Exception: pass
if _fs_pairs:
    _fs_embs = _fs_embedder.encode([p for p, _ in _fs_pairs], convert_to_numpy=True)
print(f'Bank: {len(_fs_pairs)} MMLU examples.')
_fs_checkpoint = (_fs_pairs.copy(), _fs_embs.copy())

def _fs_retrieve(query, top_k=3):
    if not _fs_pairs: return []
    q    = _fs_embedder.encode([query], convert_to_numpy=True)
    sims = (_fs_embs / (np.linalg.norm(_fs_embs, axis=1, keepdims=True) + 1e-8)) @            (q / (np.linalg.norm(q) + 1e-8)).T
    return [_fs_pairs[i] for i in np.argsort(sims[:, 0])[::-1][:top_k]]

def semantic_few_shot(question):
    opts   = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    ex     = _fs_retrieve(question.text)
    ex_blk = '\n'.join(f'Question: {q}\nAnswer: {a}\n' for q, a in ex)
    prompt = (f'Here are similar questions with correct answers:\n\n{ex_blk}\n'
              f'Now answer this. Reply with ONLY the letter (A, B, C or D).\n\n'
              f'Question: {question.text}\n{opts}\nAnswer:') if ex else (
              f'Answer this. Reply with ONLY the letter.\n\nQuestion: {question.text}\n\n{opts}\n\nAnswer:')
    raw = _llm(prompt, max_tokens=10)
    for ch in reversed(raw.strip().upper()):
        if ch in LABELS: return question.options[LABELS.index(ch)].id
    return random.choice(question.options).id

def _fs_reset():
    global _fs_pairs, _fs_embs
    _fs_pairs = list(_fs_checkpoint[0]); _fs_embs = _fs_checkpoint[1].copy()

def _fs_update(question, chosen_text, result):
    global _fs_pairs, _fs_embs
    if result.correct:
        _fs_pairs.append((question.text, chosen_text))
        _fs_embs = np.vstack([_fs_embs, _fs_embedder.encode([question.text], convert_to_numpy=True)])

run_strategy('semantic-few-shot', semantic_few_shot,
             after_answer=_fs_update, before_game=_fs_reset, comp_ids=[3])


## Strategy 4 — Hybrid-Math v1 (LLM classifier)

**Key insight (first hybrid attempt).** Some questions are definitional ("what is a sampling distribution?") and some are computational ("compute 20×−8 in Z₂₆"). For computational ones, generating and running Python code is more reliable than language-model arithmetic. We add a classification step: ask the LLM whether the question needs `CALCULATE` or `REASON`.

**Why classification seemed like a good idea.** A single universal strategy cannot be optimal for both question types. Matching the approach to the question type is a sound engineering principle.

**Why it failed (63.8%).** The LLM classifier has a strong prior toward `REASON` — it says `CALCULATE` for fewer than 20% of questions, even on clearly numerical ones. Reasons:
- The instruction-tuned model is trained to reason verbally, not to dispatch to code.
- The classifier adds an entire extra LLM call with no reliable signal.
- When the code path IS taken, the reasoning fallback has only 300 tokens — frequently truncated mid-sentence.

**What we learned.** The classifier is the bottleneck. We should remove the gating decision entirely and *always* try the code path first. This becomes the foundation of every subsequent Hybrid strategy.


In [ ]:
# ── STRATEGY 4 — Hybrid-Math v1 (LLM classifier) ─────────────────────────
_colab_pull('llama3.1:8b')
import re, subprocess, tempfile, os, random

_CLASSIFY_PROMPT = """Decide how to best answer the following question.
Reply with ONLY ONE word:
- CALCULATE  if it needs a numerical computation / formula.
- REASON     if it is conceptual / theoretical.

Question: {question}

Answer (one word):"""

_CODE_PROMPT_V1 = """Solve this math problem by writing executable Python code.
Rules:
- Output ONLY a ```python ... ``` block, nothing else.
- You may import: math, sympy, itertools, fractions, numpy, scipy.
- The LAST line must be a print() that outputs a single numerical value.

Problem: {question}
Options (context only): {options}

```python"""

_REASON_PROMPT_V1 = """Reason step by step, then on the LAST line write exactly: Answer: X
(X is A, B, C or D)

Question: {question}

{options}"""

_CODE_HEADER = (
    "from functools import reduce\nfrom itertools import combinations, permutations, product\n"
    "from fractions import Fraction\nimport math, sympy\nimport numpy as np\nfrom scipy import stats\n\n"
)

def _extract_code(raw):
    fence = re.search(r'```(?:python)?\s*\n?(.*?)```', raw, re.DOTALL)
    return fence.group(1).strip() if fence else raw.strip()

def _run_code(code, timeout=8):
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(_CODE_HEADER + code); tmp = f.name
        r = subprocess.run(['python3', tmp], capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip() or None
    except Exception: return None
    finally:
        try: os.unlink(tmp)
        except: pass

def _parse_option_value(text):
    try: return float(text.strip())
    except: pass
    nums = re.findall(r'-?\d[\d,]*\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else None

def _match_option(output, question):
    try: val = float(output.splitlines()[-1].replace(',', ''))
    except: return None
    best_id, best_diff = None, float('inf')
    for opt in question.options:
        v = _parse_option_value(opt.text)
        if v is None: continue
        d = abs(val - v) / (abs(v) + 1e-9)
        if d < best_diff: best_diff, best_id = d, opt.id
    return best_id if best_diff < 0.01 else None

def _to_letter(text):
    m = re.search(r'Answer\s*:\s*([A-D])', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    letters = re.findall(r'\b([A-D])\b', text.upper())
    return letters[-1] if letters else 'A'

def hybrid_math_v1(question):
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    resp     = _llm(_CLASSIFY_PROMPT.format(question=question.text), max_tokens=10)
    is_calc  = 'CALCULATE' in resp.strip().upper()
    if SHOW_QA: print(f'   [classifier] {"CALCULATE" if is_calc else "REASON"}')

    if is_calc:
        raw    = _llm(_CODE_PROMPT_V1.format(question=question.text, options=opts_str),
                      max_tokens=600, model=LLAMA)
        output = _run_code(_extract_code(raw))
        if output:
            matched = _match_option(output, question)
            if matched: return matched

    # Reason path (v1: tight 300-token budget, often truncated)
    resp = _llm(_REASON_PROMPT_V1.format(question=question.text, options=opts_str), max_tokens=300)
    return question.options[LABELS.index(_to_letter(resp))].id

run_strategy('hybrid-math-v1-classifier', hybrid_math_v1, comp_ids=[3])


## Strategy 5 — Hybrid-Math Llama

**Core change from S4.** Remove the LLM classifier. Instead, **always** attempt to generate and execute Python code first. This eliminates the unreliable dispatching step.

**Three-step decision logic:**

```
1. Generate Python code (Llama) → run it in a subprocess
   ├─ Output matches an option exactly → return it ✓
   ├─ Output is a plausible number (within option range) → use as hint in CoT step
   └─ Output is implausible / error → skip hint
2. CoT reasoning (with or without numerical hint)
3. Parse Answer: X from the last line
```

**Plausibility check (critical detail).** If the code output is wildly out of range (e.g., the options are 1–10 and the code outputs 10⁶), we discard it before feeding it to the CoT step — otherwise it poisons the reasoning. We check: is the computed value within `[min_option − spread, max_option + spread]`?

**Why this works.** For the ~40% of questions that are purely numerical, the code path finds the exact answer with near-perfect reliability. For the rest, the reasoning fallback performs at least as well as CoT (S2). The two paths complement each other perfectly.

**What we found.** **83.0% accuracy, 21.5 s/question.** Best single-model result. The 21 seconds is almost entirely the code generation time — the subprocess itself takes <1 s.

**Remaining weakness.** The 21 s average is cutting it close to the 30-second limit. And Llama-3.1-8B is not the strongest available model for mathematical reasoning — could a better model do better?


In [ ]:
# ── STRATEGY 5 — Hybrid-Math Llama ──────────────────────────────────────
_colab_pull('llama3.1:8b')
import re, subprocess, tempfile, os, random

_CODE_PROMPT_HM = """Solve this math problem by writing executable Python code.
When using sympy.solve(), always use solution[0] or solution[-1], never a fixed index.

Rules:
- Output ONLY a ```python ... ``` block, nothing else.
- You may import: math, sympy, itertools, fractions, numpy, scipy.
- The LAST line must be a print() that outputs a single numerical value.
- Do NOT print labels or extra text, only the raw value.

Problem: {question}
Options (context only): {options}

```python"""

_REASON_PROMPT_HM = """Answer concisely. Max 4 lines of reasoning. Do not repeat yourself.
If unsure, commit to the most likely answer anyway.
On the LAST line write exactly: Answer: X  (X is A, B, C or D - nothing else after it)

Question: {question}

{options}"""

_CODE_HEADER = (
    "from functools import reduce\nfrom itertools import combinations, permutations, product\n"
    "from fractions import Fraction\nimport math, sympy\nimport numpy as np\nfrom scipy import stats\n\n"
)

def _extract_code(raw):
    fence = re.search(r'```(?:python)?\s*\n?(.*?)```', raw, re.DOTALL)
    return fence.group(1).strip() if fence else raw.strip()

def _run_code(code, timeout=8):
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(_CODE_HEADER + code); tmp = f.name
        r = subprocess.run(['python3', tmp], capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip() or None
    except Exception: return None
    finally:
        try: os.unlink(tmp)
        except: pass

def _parse_opt_val(text):
    text = text.strip()
    try: return float(text)
    except: pass
    for p in [r'\\frac\{(-?\d+)\}\{(-?\d+)\}', r'(-?\d+)\s*/\s*(\d+)']:
        m = re.search(p, text)
        if m:
            try: return float(m.group(1)) / float(m.group(2))
            except: pass
    nums = re.findall(r'-?\d[\d,]*\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else None

def _match_option(output, question):
    try: val = float(output.splitlines()[-1].replace(',', ''))
    except: return None
    best_id, best_diff = None, float('inf')
    for opt in question.options:
        v = _parse_opt_val(opt.text)
        if v is None: continue
        d = abs(val - v) / (abs(v) + 1e-9)
        if d < best_diff: best_diff, best_id = d, opt.id
    return best_id if best_diff < 0.01 else None

def _is_plausible(output, question):
    try: val = float(output.splitlines()[-1].replace(',', ''))
    except: return False
    vs = [_parse_opt_val(o.text) for o in question.options]
    vs = [v for v in vs if v is not None]
    if not vs: return False
    lo, hi = min(vs), max(vs)
    spread = hi - lo
    return (lo - spread) <= val <= (hi + spread)

def _to_letter(text):
    m = re.search(r'Answer\s*:\s*([A-D])', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    letters = re.findall(r'\b([A-D])\b', text.upper())
    return letters[-1] if letters else 'A'

def hybrid_math_llama(question):
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))

    # Step 1: always try code first
    raw    = _llm(_CODE_PROMPT_HM.format(question=question.text, options=opts_str),
                  max_tokens=600, model=LLAMA)
    output = _run_code(_extract_code(raw))
    if SHOW_QA and output: print(f'   [code output] {output}')

    if output:
        matched = _match_option(output, question)
        if matched: return matched
        # Step 2: pass plausible hint to CoT
        if _is_plausible(output, question):
            augmented = (_REASON_PROMPT_HM.format(question=question.text, options=opts_str)
                         + f'\n\nHint: a calculation gave: {output}')
            resp = _llm(augmented, max_tokens=300)
            if SHOW_QA:
                reasoning = re.sub(r'(?is)\n?\s*answer\s*[:=].*$', '', resp).strip()
                for line in reasoning.splitlines():
                    if line.strip(): print(f'   {line.strip()}')
            return question.options[LABELS.index(_to_letter(resp))].id

    # Step 3: pure reasoning fallback
    resp = _llm(_REASON_PROMPT_HM.format(question=question.text, options=opts_str), max_tokens=600)
    if SHOW_QA:
        reasoning = re.sub(r'(?is)\n?\s*answer\s*[:=].*$', '', resp).strip()
        for line in reasoning.splitlines():
            if line.strip(): print(f'   {line.strip()}')
    return question.options[LABELS.index(_to_letter(resp))].id

run_strategy('hybrid-math-llama', hybrid_math_llama, comp_ids=[3])


## Strategy 6 — DeepSeek-R1 7B ⚠️ exceeds 30 s

**Motivation.** DeepSeek-R1 is a reasoning-first model that generates an explicit `<think>...</think>` block before answering. On math benchmarks (MATH, GSM8K), it outperforms same-size instruction-tuned models. If it is accurate *and* fast enough, it could replace our hybrid pipeline entirely.

**What we tested.** Direct application of DeepSeek-R1-7B to our benchmark with a simple prompt asking for a final `Answer: X` line after thinking.

**What we found.** 68.1% accuracy, **45.3 s/question** — well over the 30-second game limit. The reasoning quality is genuinely better than Llama or Mistral, but the model generates very long thinking traces (often 500–1500 tokens) before outputting the answer.

**What we learned — two lessons:**
1. **Thinking token budget is the real bottleneck**, not model quality. A model that thinks for 45 seconds is useless in a 30-second game regardless of accuracy.
2. **The thinking paradigm is promising.** If we could limit the thinking budget, or use a smaller/faster reasoning model, the quality benefit might be preserved. This motivates testing DeepSeek-R1-1.5B (Strategy 7).


In [ ]:
# ── STRATEGY 6 — DeepSeek-R1 7B (EXCEEDS 30 s LIMIT) ────────────────────
_colab_pull('deepseek-r1:7b')
import re, random, time

_DS7B_MODEL = 'deepseek-r1:7b'

def _ds7b_parse(raw, question):
    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    if not clean or '<think>' in clean:
        lines = [l.strip() for l in raw.splitlines() if l.strip()]
        clean = '\n'.join(lines[-3:])
    upper = clean.upper()
    m = re.search(r'ANSWER\s*[:=]\s*([A-D])', upper)
    if m: return question.options[LABELS.index(m.group(1))].id
    for ch in reversed(upper):
        if ch in LABELS: return question.options[LABELS.index(ch)].id
    return random.choice(question.options).id

def deepseek_7b(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    t0   = time.time()
    raw  = _llm(
        f'Answer this multiple choice question.\n'
        f'Think carefully, then on the LAST line write exactly: Answer: X\n'
        f'(X is A, B, C or D)\n\nQuestion: {question.text}\n\n{opts}',
        max_tokens=2000, model=_DS7B_MODEL,
    )
    dt = time.time() - t0
    if SHOW_QA:
        think = re.search(r'<think>(.*?)</think>', raw, re.DOTALL)
        content = think.group(1).strip()[:300] if think else ''
        if content: print(f'   [thinking] {content[:200]}...')
    print(f'   [time] {dt:.1f}s {"<-- OVER 30s LIMIT" if dt > 30 else "OK"}')
    return _ds7b_parse(raw, question)

run_strategy('deepseek-7b', deepseek_7b, comp_ids=[3])


## Strategy 7 — DeepSeek-R1 1.5B

**Motivation.** Strategy 6 showed that the 7B reasoning model is too slow. The 1.5B distilled variant generates much shorter thinking traces — can it keep the quality while fitting in 30 seconds?

**What we found.** 51.1% accuracy, 7.4 s/question. Fast enough, but accuracy is *below the zero-shot baseline*. Two problems:
1. **Chronic truncation.** At 1.5B parameters the model nearly always runs out of tokens mid-thought — the `</think>` tag never appears. We fall back to scanning the last lines of the raw output for a letter, which is noisy.
2. **Weak reasoning capacity.** At 1.5B, the model does not have enough parameters to reliably solve university-level math.

**What we learned.** Pure 1.5B reasoning is not viable as a standalone strategy. But the model has a hidden strength: when given a *concrete numerical hint* and asked to pick the closest option, it performs adequately — it only needs to map a number to a letter, not derive the number itself. This motivates a hybrid approach (Strategy 8): use a strong model to generate the hint and DeepSeek-1.5B only to pick the answer.


In [ ]:
# ── STRATEGY 7 — DeepSeek-R1 1.5B ───────────────────────────────────────
_colab_pull('deepseek-r1:1.5b')
import re, random, time

_DS15_MODEL = 'deepseek-r1:1.5b'

def _ds15_parse(raw, question):
    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    if not clean or '<think>' in clean:
        if SHOW_QA: print('   [parser] thinking truncated, scanning last lines')
        lines = [l.strip() for l in raw.splitlines() if l.strip()]
        clean = '\n'.join(lines[-3:])
    upper = clean.upper()
    m = re.search(r'ANSWER\s*[:=]\s*([A-D])', upper)
    if m: return question.options[LABELS.index(m.group(1))].id
    for ch in reversed(upper):
        if ch in LABELS: return question.options[LABELS.index(ch)].id
    return random.choice(question.options).id

def deepseek_1_5b(question):
    opts = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))
    t0   = time.time()
    raw  = _llm(
        f'Answer this multiple choice question.\n'
        f'Think briefly, then on the LAST line write exactly: Answer: X\n'
        f'(X is A, B, C or D)\n\nQuestion: {question.text}\n\n{opts}',
        max_tokens=1000, model=_DS15_MODEL,
    )
    print(f'   [time] {time.time()-t0:.1f}s')
    return _ds15_parse(raw, question)

run_strategy('deepseek-1.5b', deepseek_1_5b, comp_ids=[3])


## Strategy 8 — Hybrid-Math DeepSeek

**Hypothesis from S7.** DeepSeek-1.5B fails when it must derive the answer from scratch, but might succeed when given a pre-computed numerical hint. So: use **Llama** (proven code generator from S5) for computation, then hand the result to **DeepSeek-1.5B** only for the final "which option is closest?" step.

**Expected benefit.** DeepSeek's chain-of-thought reasoning, even at 1.5B, should be better at interpreting a numerical hint than Llama's vanilla CoT — giving us a net improvement over S5.

**What we found.** 59.6% accuracy, 11.0 s/question. Worse than S5 (83%) and even worse than pure CoT (70%).

**Post-mortem.** The problem is the *fallback path*. When code fails (which happens on ~40% of questions), we fall back to pure DeepSeek-1.5B reasoning — which is weaker than Llama's reasoning. So we traded Llama's strong reasoning for DeepSeek's weak reasoning, and the gain from the hint never compensates.

**What we learned (critical).** The choice of *reasoning model* matters enormously in the fallback path, not just the code generation model. The two roles should ideally be filled by the *same* capable model. This directly motivates Strategy 9: what if we used `qwen2.5:7b` — the strongest single model we benchmarked (S11) — for both code generation and reasoning?


In [ ]:
# ── STRATEGY 8 — Hybrid-Math DeepSeek ───────────────────────────────────
_colab_pull('llama3.1:8b', 'deepseek-r1:1.5b')
import re, subprocess, tempfile, os, random

_DS15_MODEL = 'deepseek-r1:1.5b'

_CODE_PROMPT_DS = """Solve this math problem by writing executable Python code.
When using sympy.solve(), always use solution[0] or solution[-1], never a fixed index.

Rules:
- Output ONLY a ```python ... ``` block, nothing else.
- You may import: math, sympy, itertools, fractions, numpy, scipy.
- The LAST line must be a print() that outputs a single numerical value.

Problem: {question}
Options (context only): {options}

```python"""

_DS_REASON_PROMPT = """Answer this multiple choice question.
Think carefully, then on the LAST line write exactly: Answer: X
(X is A, B, C or D)

Question: {question}

{options}"""

_CODE_HEADER = (
    "from functools import reduce\nfrom itertools import combinations, permutations, product\n"
    "from fractions import Fraction\nimport math, sympy\nimport numpy as np\nfrom scipy import stats\n\n"
)

def _extract_code(raw):
    fence = re.search(r'```(?:python)?\s*\n?(.*?)```', raw, re.DOTALL)
    return fence.group(1).strip() if fence else raw.strip()

def _run_code(code, timeout=8):
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(_CODE_HEADER + code); tmp = f.name
        r = subprocess.run(['python3', tmp], capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip() or None
    except Exception: return None
    finally:
        try: os.unlink(tmp)
        except: pass

def _parse_opt_val(text):
    try: return float(text.strip())
    except: pass
    nums = re.findall(r'-?\d[\d,]*\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else None

def _match_option(output, question):
    try: val = float(output.splitlines()[-1].replace(',', ''))
    except: return None
    best_id, best_diff = None, float('inf')
    for opt in question.options:
        v = _parse_opt_val(opt.text)
        if v is None: continue
        d = abs(val - v) / (abs(v) + 1e-9)
        if d < best_diff: best_diff, best_id = d, opt.id
    return best_id if best_diff < 0.01 else None

def _is_plausible(output, question):
    try: val = float(output.splitlines()[-1].replace(',', ''))
    except: return False
    vs = [_parse_opt_val(o.text) for o in question.options]
    vs = [v for v in vs if v is not None]
    if not vs: return False
    lo, hi = min(vs), max(vs); spread = hi - lo
    return (lo - spread) <= val <= (hi + spread)

def _ds15_parse_letter(raw, question):
    clean = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    if not clean or '<think>' in clean:
        lines = [l.strip() for l in raw.splitlines() if l.strip()]
        clean = '\n'.join(lines[-3:])
    m = re.search(r'ANSWER\s*[:=]\s*([A-D])', clean.upper())
    if m: return question.options[LABELS.index(m.group(1).upper())].id
    for ch in reversed(clean.upper()):
        if ch in 'ABCD': return question.options[LABELS.index(ch)].id
    return question.options[0].id

def _to_letter(text):
    m = re.search(r'Answer\s*:\s*([A-D])', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    letters = re.findall(r'\b([A-D])\b', text.upper())
    return letters[-1] if letters else 'A'

def hybrid_math_deepseek(question):
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))

    raw    = _llm(_CODE_PROMPT_DS.format(question=question.text, options=opts_str),
                  max_tokens=600, model=LLAMA)
    output = _run_code(_extract_code(raw))
    if SHOW_QA and output: print(f'   [code output] {output}')

    if output:
        matched = _match_option(output, question)
        if matched: return matched
        if _is_plausible(output, question):
            augmented = (_DS_REASON_PROMPT.format(question=question.text, options=opts_str)
                         + f'\n\nHint: a calculation gave: {output}')
            raw_ds = _llm(augmented, max_tokens=1000, model=_DS15_MODEL)
            return _ds15_parse_letter(raw_ds, question)

    raw = _llm(_DS_REASON_PROMPT.format(question=question.text, options=opts_str),
               max_tokens=1000, model=_DS15_MODEL)
    return _ds15_parse_letter(raw, question)

run_strategy('hybrid-math-deepseek', hybrid_math_deepseek, comp_ids=[3])


## Strategy 9 — Hybrid-Math Qwen

**Decisive question.** Strategy 5 (Hybrid-Math Llama, 83%) uses `llama3.1:8b` for both code generation and reasoning. Strategy 11 (Multi-Model Benchmark) shows that `qwen2.5:7b` is the strongest single-model reasoner on our benchmark at ~70% for pure CoT. Does swapping Llama for Qwen in the hybrid pipeline give a better result?

**Architecture.** Identical to Strategy 5 — always try code first, plausibility check, hint-augmented CoT fallback — but with `qwen2.5:7b` everywhere instead of `llama3.1:8b`.

**Why Qwen might be better:**
- Qwen2.5-7B is trained on a larger math corpus than Llama-3.1-8B.
- Qwen generates cleaner Python code with fewer syntax errors.
- Qwen's CoT reasoning on the fallback path is stronger (confirmed by S11).

**What we found.** **81% accuracy (38/47), avg 9.6 s/question.**

This is remarkable: nearly as good as Hybrid-Math Llama (83%) but **more than twice as fast** (9.6 s vs 21.5 s). The speed improvement comes from Qwen generating more concise code — fewer tokens, faster generation.

**Key observations from the run:**
- Code path succeeds and matches option directly on ~45% of questions (ring arithmetic, LCM/GCD, geometry, probability)
- Hint-augmented CoT handles a further ~25% (questions where code produces a plausible but non-matching number)
- Pure reasoning fallback handles the remaining ~30% (conceptual statistics, sampling theory)
- Errors concentrate on questions requiring *creative mathematical formulation* (e.g., computing work along a path) where Qwen's generated code is mathematically correct but answers the wrong sub-question

**Conclusion.** Hybrid-Math Qwen is arguably our best *practical* strategy: it is 2× faster than Hybrid-Math Llama while sacrificing only 2 percentage points of accuracy, leaving a 20-second buffer from the game limit.


In [ ]:
# ── STRATEGY 9 — Hybrid-Math Qwen ────────────────────────────────────────
_colab_pull('qwen2.5:7b')
import re, subprocess, tempfile, os, random

_CODE_PROMPT_HMQ = """Solve this math problem by writing executable Python code.
When using sympy.solve(), always use solution[0] or solution[-1], never a fixed index.

Rules:
- Output ONLY a ```python ... ``` block, nothing else.
- You may import: math, sympy, itertools, fractions, numpy, scipy.
- The LAST line must be a print() that outputs a single numerical value.
- Do NOT print labels or extra text, only the raw value.

Problem: {question}
Options (context only): {options}

```python"""

_REASON_PROMPT_HMQ = """Answer concisely. Max 4 lines of reasoning. Do not repeat yourself.
If unsure, commit to the most likely answer anyway.
On the LAST line write exactly: Answer: X  (X is A, B, C or D - nothing else after it)

Question: {question}

{options}"""

_CODE_HEADER = (
    "from functools import reduce\nfrom itertools import combinations, permutations, product\n"
    "from fractions import Fraction\nimport math, sympy\nimport numpy as np\nfrom scipy import stats\n\n"
)

def _extract_code(raw):
    fence = re.search(r'```(?:python)?\s*\n?(.*?)```', raw, re.DOTALL)
    return fence.group(1).strip() if fence else raw.strip()

def _run_code(code, timeout=8):
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(_CODE_HEADER + code); tmp = f.name
        r = subprocess.run(['python3', tmp], capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip() or None
    except Exception: return None
    finally:
        try: os.unlink(tmp)
        except: pass

def _parse_opt_val(text):
    text = text.strip()
    try: return float(text)
    except: pass
    for p in [r'\\frac\{(-?\d+)\}\{(-?\d+)\}', r'(-?\d+)\s*/\s*(\d+)']:
        m = re.search(p, text)
        if m:
            try: return float(m.group(1)) / float(m.group(2))
            except: pass
    nums = re.findall(r'-?\d[\d,]*\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else None

def _match_option(output, question):
    try: val = float(output.splitlines()[-1].replace(',', ''))
    except: return None
    best_id, best_diff = None, float('inf')
    for opt in question.options:
        v = _parse_opt_val(opt.text)
        if v is None: continue
        d = abs(val - v) / (abs(v) + 1e-9)
        if d < best_diff: best_diff, best_id = d, opt.id
    return best_id if best_diff < 0.01 else None

def _is_plausible(output, question):
    try: val = float(output.splitlines()[-1].replace(',', ''))
    except: return False
    vs = [_parse_opt_val(o.text) for o in question.options]
    vs = [v for v in vs if v is not None]
    if not vs: return False
    lo, hi = min(vs), max(vs); spread = hi - lo
    return (lo - spread) <= val <= (hi + spread)

def _to_letter(text):
    m = re.search(r'Answer\s*:\s*([A-D])', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    letters = re.findall(r'\b([A-D])\b', text.upper())
    return letters[-1] if letters else 'A'

def hybrid_math_qwen(question):
    opts_str = '\n'.join(f'{LABELS[i]}) {opt.text}' for i, opt in enumerate(question.options))

    raw    = _llm(_CODE_PROMPT_HMQ.format(question=question.text, options=opts_str),
                  max_tokens=600, model=QWEN)
    output = _run_code(_extract_code(raw))
    if SHOW_QA and output: print(f'   [code output] {output}')

    if output:
        matched = _match_option(output, question)
        if matched: return matched
        if _is_plausible(output, question):
            augmented = (_REASON_PROMPT_HMQ.format(question=question.text, options=opts_str)
                         + f'\n\nHint: a calculation gave: {output}')
            resp = _llm(augmented, max_tokens=300, model=QWEN)
            if SHOW_QA:
                for line in re.sub(r'(?is)\n?\s*answer\s*[:=].*$', '', resp).strip().splitlines():
                    if line.strip(): print(f'   {line.strip()}')
            return question.options[LABELS.index(_to_letter(resp))].id

    resp = _llm(_REASON_PROMPT_HMQ.format(question=question.text, options=opts_str),
                max_tokens=600, model=QWEN)
    if SHOW_QA:
        for line in re.sub(r'(?is)\n?\s*answer\s*[:=].*$', '', resp).strip().splitlines():
            if line.strip(): print(f'   {line.strip()}')
    return question.options[LABELS.index(_to_letter(resp))].id

run_strategy('hybrid-math-qwen', hybrid_math_qwen, comp_ids=[3])


## Strategy 10 — Prompt Engineering

**Motivation.** Before building the ensemble (S12), we want to know which *instruction framing* extracts the best answers from `qwen2.5:7b`. Different phrasing can shift accuracy by several points, and the best prompt becomes the foundation of the ensemble.

**Design.** We test 10 prompt styles, all using the same two-line output format (`Answer: X / Confidence: Y`) so results are comparable and the confidence score can be used later for weighted voting.

| Variant | Core instruction |
|---------|-----------------|
| `baseline` | Reason internally, then Answer + Confidence |
| `direct` | No reasoning shown, two lines only |
| `elimination` | Eliminate clearly wrong options first |
| `self_check` | Solve, then verify against each option |
| `equation_first` | Rewrite as equations, then solve |
| `estimate_compute` | Estimate magnitude, then compute precisely |
| `units_constraints` | Track units and domain restrictions |
| `two_path_consistency` | Try two independent solution paths |
| `strict_elimination` | Evaluate each option A-D explicitly |
| `anti_trick` | Watch for sign errors, rounding traps |

**What we found.** The `baseline` variant (which matches the prompt used throughout our benchmarks) performs comparably to most others. The differences between variants are small (~2–5 pp), suggesting the model's fundamental capability is the bottleneck — not the prompt framing. The best variant is registered in RESULTS.

**What we learned.** Prompt engineering has diminishing returns once you are already asking for explicit reasoning. The bigger gains come from tool use (code execution, S5/S9) and model selection (S11), not from rewording the instruction.


In [ ]:
# ── STRATEGY 10 — Prompt Engineering ────────────────────────────────────
_colab_pull('qwen2.5:7b')
import re, random
from collections import Counter

_PE_MODEL = 'qwen2.5:7b'

def _fmt_opts(q): return '\n'.join(f'{LABELS[i]}) {o.text}' for i, o in enumerate(q.options))

def _el(text):
    if not text: return None
    up = text.strip().upper()
    for pat in [r'ANSWER\s*[:=\-]?\s*([ABCD])', r'FINAL\s+ANSWER\s*[:=]?\s*([ABCD])',
                r'\(([ABCD])\)', r'\b([ABCD])\b']:
        m = re.search(pat, up)
        if m: return m.group(1)
    lines = [l.strip() for l in up.splitlines() if l.strip()]
    if lines:
        last = re.sub(r'[^ABCD]', '', lines[-1])
        if len(last) == 1: return last
    return None

def _ec(text):
    if not text: return None
    up = text.strip().upper().replace(',', '.')
    m = re.search(r'CONFIDENCE\s*[:=]\s*([0-9]*\.?[0-9]+)', up)
    if m:
        try:
            v = float(m.group(1))
            return min(1.0, v / 100.0 if v > 1.0 else v)
        except: pass
    return None

def _ask_pe(prompt, model=_PE_MODEL):
    raw    = _llm(prompt, max_tokens=256, model=model)
    letter = _el(raw)
    conf   = _ec(raw)
    if letter is None:
        raw2   = _llm(prompt + '\n\nOutput ONLY two lines:\nAnswer: X\nConfidence: Y',
                      max_tokens=40, model=model)
        letter = _el(raw2) or random.choice(LABELS)
        if conf is None: conf = _ec(raw2)
    return letter, conf if conf is not None else 0.5

_PROMPT_VARIANTS = {
    'baseline':             lambda q: f'You are solving a MCQ on mathematics. Do all reasoning internally.\nReturn exactly:\nAnswer: X\nConfidence: Y (Y in [0,1]).\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'direct':               lambda q: f'Answer the MCQ. No reasoning shown.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'elimination':          lambda q: f'Eliminate clearly wrong options internally. Return:\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'self_check':           lambda q: f'Solve then self-check against each option.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'equation_first':       lambda q: f'First rewrite as equations. Then solve and map to options.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'estimate_compute':     lambda q: f'Estimate magnitude first to rule out options. Then compute.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'units_constraints':    lambda q: f'Track units and domain constraints. Reject violating options.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'two_path_consistency': lambda q: f'Try two independent solution paths. Reconcile if they disagree.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'strict_elimination':   lambda q: f'Evaluate options A, B, C, D one by one. Eliminate explicitly.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
    'anti_trick':           lambda q: f'Watch for sign errors, parentheses, rounding traps, boundary conditions.\nAnswer: X\nConfidence: Y.\n\nQuestion: {q.text}\n\n{_fmt_opts(q)}\n',
}

if BENCHMARK is not None:
    _pe_scores = {}
    for _vname, _vfn in _PROMPT_VARIANTS.items():
        _correct = 0
        for _q in BENCHMARK:
            _letter, _ = _ask_pe(_vfn(_q))
            _correct  += int(_letter == LABELS[_q.correct_id])
        _pe_scores[_vname] = _correct / len(BENCHMARK)
        print(f'  {_vname:28s} {_pe_scores[_vname]:.0%}')

    import matplotlib.pyplot as plt, numpy as np
    _names, _vals = list(_pe_scores.keys()), list(_pe_scores.values())
    _fig, _ax = plt.subplots(figsize=(11, 4))
    _ax.bar(range(len(_names)), _vals, color='steelblue')
    _ax.set_xticks(range(len(_names))); _ax.set_xticklabels(_names, rotation=25, ha='right')
    _ax.set_ylabel('Accuracy'); _ax.set_ylim(0, 1)
    _ax.set_title(f'Prompt variant comparison — {_PE_MODEL}')
    for i, v in enumerate(_vals): _ax.text(i, v+.01, f'{v:.0%}', ha='center', fontsize=8)
    _ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

    BEST_PROMPT_NAME = max(_pe_scores, key=_pe_scores.get)
    BEST_PROMPT_FN   = _PROMPT_VARIANTS[BEST_PROMPT_NAME]
    print(f'\nBest prompt: {BEST_PROMPT_NAME} ({_pe_scores[BEST_PROMPT_NAME]:.0%})')
else:
    print('Load benchmark first (load_benchmark()) to compare prompt variants.')
    BEST_PROMPT_NAME = 'baseline'
    BEST_PROMPT_FN   = _PROMPT_VARIANTS['baseline']

def _best_prompt_strategy(question):
    letter, _ = _ask_pe(BEST_PROMPT_FN(question))
    return question.options[LABELS.index(letter)].id

run_strategy(f'prompt-{BEST_PROMPT_NAME}', _best_prompt_strategy, comp_ids=[3])


## Strategy 11 — Multi-Model Benchmark

**Purpose.** Compare `qwen2.5:7b`, `mistral:7b`, and `llama3.1:8b` side by side on the same prompt to find the best single model for use in the ensemble. This is also the empirical basis for the decision to use Qwen in Strategy 9.

**Method.** For each model, run the `equation_first` prompt (best from S10) on all 47 benchmark questions. Measure accuracy and average response time.

**What we found.** `qwen2.5:7b` achieves the highest accuracy (~70%) with the lowest response time (~1 s/question). `mistral:7b` and `llama3.1:8b` score lower. Qwen's advantage on maths is consistent with its pre-training: Qwen2.5 was trained with a much larger mathematics corpus than Llama-3.1 or Mistral at the same parameter count.

**Why this matters for the ensemble (S12).** If one model is reliably better, it should have more weight in the vote. This motivates the confidence-weighted voting scheme: Qwen will naturally produce higher confidence values on questions it understands, so the weighting partially corrects for model quality differences.

**Critical observation.** Even the best single model (70%) is far below the Hybrid-Math strategies (81–83%). This tells us that the code execution step in S5/S9 — not the model choice — is the dominant factor in accuracy. The ensemble in S12 deliberately abandons code execution and uses pure CoT, which is why it cannot match the hybrid strategies.


In [ ]:
# ── STRATEGY 11 — Multi-Model Benchmark ─────────────────────────────────
_colab_pull('qwen2.5:7b', 'mistral:7b', 'llama3.1:8b')
import re, random, time as _time, numpy as np, matplotlib.pyplot as plt

_BENCH_MODELS = ['qwen2.5:7b', 'mistral:7b', 'llama3.1:8b']

def _fmt_opts(q): return '\n'.join(f'{LABELS[i]}) {o.text}' for i, o in enumerate(q.options))

def _el(text):
    if not text: return None
    up = text.strip().upper()
    for pat in [r'ANSWER\s*[:=\-]?\s*([ABCD])', r'\(([ABCD])\)', r'\b([ABCD])\b']:
        m = re.search(pat, up)
        if m: return m.group(1)
    lines = [l.strip() for l in up.splitlines() if l.strip()]
    if lines:
        last = re.sub(r'[^ABCD]', '', lines[-1])
        if len(last) == 1: return last
    return None

def _ask_model(question, model):
    opts   = _fmt_opts(question)
    prompt = (f'Answer this multiple-choice question. Do all reasoning internally.\n'
              f'Return exactly two final lines:\nAnswer: X\nConfidence: Y (Y in [0,1]).\n\n'
              f'Question: {question.text}\n\n{opts}\n')
    raw    = _llm(prompt, max_tokens=256, model=model)
    letter = _el(raw)
    if letter is None:
        raw2   = _llm(prompt + '\n\nOutput ONLY: Answer: X\nConfidence: Y', max_tokens=30, model=model)
        letter = _el(raw2) or random.choice(LABELS)
    return letter

if BENCHMARK is not None:
    _bm_scores, _bm_times = {}, {}
    for _m in _BENCH_MODELS:
        _correct, _ts = 0, []
        for _q in BENCHMARK:
            _t0 = _time.time()
            _letter = _ask_model(_q, _m)
            _ts.append(_time.time() - _t0)
            _correct += int(_letter == LABELS[_q.correct_id])
        _bm_scores[_m] = _correct / len(BENCHMARK)
        _bm_times[_m]  = sum(_ts) / len(_ts)
        print(f'  {_m:20s} acc={_bm_scores[_m]:.0%}  avg={_bm_times[_m]:.1f}s')

    _xs = np.arange(len(_BENCH_MODELS))
    _fig, (_a1, _a2) = plt.subplots(1, 2, figsize=(13, 4))
    _a1.bar(_xs, [_bm_scores[m] for m in _BENCH_MODELS], color='steelblue')
    _a1.set_title('Model accuracy'); _a1.set_ylabel('Accuracy'); _a1.set_ylim(0, 1)
    for i, m in enumerate(_BENCH_MODELS): _a1.text(i, _bm_scores[m]+.01, f'{_bm_scores[m]:.0%}', ha='center', fontsize=8)
    _a2.bar(_xs, [_bm_times[m] for m in _BENCH_MODELS], color='indianred')
    _a2.set_title('Avg response time'); _a2.set_ylabel('s')
    _a2.axhline(30, color='black', ls='--', lw=1)
    for ax in (_a1, _a2):
        ax.set_xticks(_xs); ax.set_xticklabels(_BENCH_MODELS, rotation=20, ha='right')
        ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

    BEST_MODEL = max(_bm_scores, key=_bm_scores.get)
    print(f'\nBest model: {BEST_MODEL} ({_bm_scores[BEST_MODEL]:.0%})')
else:
    print('Load benchmark first.'); BEST_MODEL = 'qwen2.5:7b'

def _best_model_strategy(question):
    return question.options[LABELS.index(_ask_model(question, BEST_MODEL))].id

run_strategy(f'model-{BEST_MODEL}', _best_model_strategy, comp_ids=[3])


## Strategy 12 — Parallel Confidence-Weighted Ensemble

**Why ensembles generally help.** When models make *independent* errors on *different* questions, majority voting reduces the error rate. If each of 3 models independently has a 70% chance of being correct, and they err independently, the majority is right ~78% of the time (3-choose-2 probability).

**Why the original implementation was too slow.**  
Ollama keeps only 1 model loaded in VRAM by default. Every model switch costs ~15 s of swap, so 3 models × sequential calls = ~45 s overhead per question alone — unplayable.

**The parallelism fix — two changes applied together:**

| Fix | Effect |
|-----|--------|
| `OLLAMA_MAX_LOADED_MODELS=3` + `OLLAMA_NUM_PARALLEL=3` (set at server start) | All 3 models stay resident in VRAM; 3 simultaneous requests allowed |
| `ThreadPoolExecutor(max_workers=3)` + pre-warm ping | All 3 requests fire at the same time; no swap on first question |

**What we found. 70% accuracy (33/47), avg 16.0 s/question.**

This is a disappointing result — worse than both Hybrid-Math strategies (81–83%). The reasons are clear from the question-by-question trace:

1. **No code execution.** The ensemble uses pure CoT prompts. The +10–20 pp that code execution adds in S5/S9 is entirely absent here. This is the dominant factor.

2. **Majority can overrule the correct model.** On several questions, Qwen answers correctly but Llama and Mistral agree on the wrong answer (2 votes vs 1), so the ensemble returns the wrong answer. Example: "Alfred asks probability" — Qwen correctly says 0.3, but Llama+Mistral outvote it.

3. **16 s average is surprisingly high.** Even with parallelism, some questions trigger the repair-pass (second LLM call) on multiple models simultaneously, which serializes on Ollama's side when all slots are busy.

**What we learned.** An ensemble of *weak CoT models* does not beat a *single model with tool use*. The lesson is that the architecture of the strategy (code + reason) matters far more than the number of models. If we were to build an ensemble of Hybrid-Math agents (each doing code + reason), performance would likely exceed 83% — but at the cost of 3× the time budget.

**Practical conclusion.** For the live game (30-second limit), Hybrid-Math Qwen (S9: 81%, 9.6 s) is a better choice than the ensemble (S12: 70%, 16 s) on every dimension.


In [ ]:
# ── STRATEGY 12 — Parallel Confidence-Weighted Ensemble ─────────────────
_colab_pull('qwen2.5:7b', 'mistral:7b', 'llama3.1:8b')
import re, random
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

_ENS_MODELS = ['qwen2.5:7b', 'mistral:7b', 'llama3.1:8b']

# ── Pre-warm: load all models into VRAM before the benchmark starts ──────
def _prewarm(models):
    print('Pre-warming models (loading into VRAM)...')
    def _ping(m):
        try: _llm('Say OK.', max_tokens=4, model=m); return m, True
        except Exception as e: return m, str(e)
    with ThreadPoolExecutor(max_workers=len(models)) as ex:
        for m, ok in ex.map(_ping, models):
            print(f'  {m}: {"ready" if ok is True else ok}')
    print('All models loaded.')

_prewarm(_ENS_MODELS)

# ── Helpers ───────────────────────────────────────────────────────────────
def _fmt_opts(q): return '\n'.join(f'{LABELS[i]}) {o.text}' for i, o in enumerate(q.options))

def _el(text):
    if not text: return None
    up = text.strip().upper()
    for pat in [r'ANSWER\s*[:=\-]?\s*([ABCD])', r'\(([ABCD])\)', r'\b([ABCD])\b']:
        m = re.search(pat, up)
        if m: return m.group(1)
    lines = [l.strip() for l in up.splitlines() if l.strip()]
    if lines:
        last = re.sub(r'[^ABCD]', '', lines[-1])
        if len(last) == 1: return last
    return None

def _ec(text):
    if not text: return None
    up = text.strip().upper().replace(',', '.')
    m = re.search(r'CONFIDENCE\s*[:=]\s*([0-9]*\.?[0-9]+)', up)
    if m:
        try:
            v = float(m.group(1))
            return min(1.0, v / 100.0 if v > 1.0 else v)
        except: pass
    return None

def _ask_with_conf(question, model):
    opts   = _fmt_opts(question)
    prompt = (f'Answer this multiple-choice question. Do all reasoning internally.\n'
              f'Return exactly two final lines:\nAnswer: X\nConfidence: Y (Y in [0,1]).\n\n'
              f'Question: {question.text}\n\n{opts}\n')
    raw    = _llm(prompt, max_tokens=256, model=model)
    letter = _el(raw)
    conf   = _ec(raw)
    if letter is None:
        raw2   = _llm(prompt + '\n\nOutput ONLY:\nAnswer: X\nConfidence: Y',
                      max_tokens=30, model=model)
        letter = _el(raw2) or random.choice(LABELS)
        if conf is None: conf = _ec(raw2)
    conf = conf if conf is not None else 0.5
    if SHOW_QA: print(f'   [{model:20s}] -> {letter}  conf={conf:.2f}')
    return model, letter, conf

def _majority_vote(votes, confs, priority):
    weights = {L: 0.0 for L in LABELS}
    counts  = Counter()
    for letter, conf in zip(votes, confs):
        if letter in LABELS:
            weights[letter] += conf; counts[letter] += 1
    if not counts: return random.choice(LABELS)
    max_w = max(weights.values())
    top   = {k for k, v in weights.items() if v == max_w and v > 0}
    if len(top) == 1: return next(iter(top))
    max_c = max(counts.get(k, 0) for k in top)
    top   = {k for k in top if counts.get(k, 0) == max_c}
    if len(top) == 1: return next(iter(top))
    for m, v in zip(priority, votes):
        if v in top: return v
    return next(iter(top))

def ensemble(question):
    model_votes, model_confs = {}, {}
    with ThreadPoolExecutor(max_workers=len(_ENS_MODELS)) as ex:
        futures = {ex.submit(_ask_with_conf, question, m): m for m in _ENS_MODELS}
        for fut in as_completed(futures):
            try:
                m, letter, conf = fut.result()
                model_votes[m] = letter; model_confs[m] = conf
            except Exception:
                m = futures[fut]
                model_votes[m] = random.choice(LABELS); model_confs[m] = 0.0
    votes  = [model_votes[m] for m in _ENS_MODELS]
    confs  = [model_confs[m] for m in _ENS_MODELS]
    winner = _majority_vote(votes, confs, _ENS_MODELS)
    if SHOW_QA: print(f'   [vote] {dict(zip(_ENS_MODELS, votes))}  -> {winner}')
    return question.options[LABELS.index(winner)].id

run_strategy('ensemble', ensemble, comp_ids=[3])


## Strategy 13 — Fine-tuned Qwen2.5-7B (LoRA / MLX)

### Why fine-tuning?

The benchmark showed that even the best prompting approach (Hybrid-Math Llama, 83%) still misses questions that a well-calibrated model would get right. Fine-tuning lets the model specialise on:
- The exact **output format** (`Answer: X / Confidence: Y` on the last two lines)
- The **domain** (university-level maths and statistics MCQs)

without changing the base weights permanently.

### Why Qwen2.5-7B-Instruct?

- Best single-model baseline on the benchmark (S11: 70%, ~1 s/question).
- 7B fits on Apple Silicon unified memory (~15 GB) — required for local MLX fine-tuning.
- Strong instruction-following: the Instruct variant already knows structured output formats.
- Qwen2.5 outperforms Llama-3.1-8B and Mistral-7B on MATH benchmarks at the same size.

### Why LoRA?

Full fine-tuning of 7B requires ~80–140 GB of GPU memory and hours of compute.  
**LoRA** freezes the base weights and adds small low-rank matrices to each attention layer (rank `r=8`, ~0.1% of total parameters):
- Training fits in ~15 GB of Apple Silicon unified memory via **MLX**.
- The adapter is only ~50 MB vs ~15 GB for a full checkpoint.
- Training converged in **500 iterations** (~10 minutes on an M-series chip).

**Training data:** MathQA + MMLU-Pro Math.  
**Prompt style:** `equation_first` — rewrite as equations, solve, output `Answer: X / Confidence: Y`.  
**Hyperparameters:** rank=8, scale=20, lr=1e-5, batch_size=1, iters=500.

### Results

| | Accuracy (47-question benchmark) |
|---|---|
| Qwen2.5-7B base (Ollama) | 70.2% |
| Qwen2.5-7B base (MLX) | 68.1% |
| Qwen2.5-7B fine-tuned (LoRA, MLX) | **76.6%** |

### Colab usage

The LoRA adapter must be **fused** into the base model first (`mlx_lm.fuse`), producing a HuggingFace-compatible checkpoint (~15 GB). Upload the fused folder to Google Drive, then set `_FT_FUSED_MODEL_COLAB` to its Drive path. The cell loads it in **4-bit** via `transformers` + `bitsandbytes`.

> **Drive mount:** uncomment the `drive.mount` line in the cell below.


In [ ]:
# ── STRATEGY 13 — Fine-tuned Qwen2.5-7B (LoRA fused) ────────────────────
import re, random, time, os, sys, zipfile

IN_COLAB  = 'google.colab' in sys.modules
_ON_COLAB = IN_COLAB
_FT_TOKENS = 96

# ── Paths ─────────────────────────────────────────────────────────────────
_FT_ZIP   = '/content/drive/MyDrive/NLP_project/fused_qwen25_mcq_package.zip'
_FT_PATH  = '/content/fused_qwen25_mcq'

# ── Unzip from Drive if not already extracted ─────────────────────────────
if _ON_COLAB:
    if not os.path.isdir(_FT_PATH):
        print('Extracting model from Drive zip...')
        if not os.path.exists(_FT_ZIP):
            raise RuntimeError(f'Zip not found at {_FT_ZIP} — check the filename on Drive.')
        os.system(f'unzip -q "{_FT_ZIP}" -d /content/')
        # the folder inside the zip may have a different name — symlink it
        with zipfile.ZipFile(_FT_ZIP) as z:
            top_dirs = {p.split('/')[0] for p in z.namelist() if '/' in p}
        for folder in top_dirs:
            candidate = f'/content/{folder}'
            if os.path.isdir(candidate) and candidate != _FT_PATH:
                os.system(f'ln -sfn "{candidate}" "{_FT_PATH}"')
                print(f'Linked {candidate} -> {_FT_PATH}')
                break
    if not os.path.isdir(_FT_PATH):
        raise RuntimeError(f'Model not found at {_FT_PATH} after extraction.')
    print(f'Model path ready: {_FT_PATH}')
    os.system(f'ls {_FT_PATH}')

# ── Prompt — identical to the one used during fine-tuning ─────────────────
def _ft_prompt(question):
    opts = '\n'.join(f'{LABELS[i]}) {o.text}' for i, o in enumerate(question.options))
    return (
        'First rewrite the problem into equations/formal expressions. '
        'Then solve and map the result to the options. '
        'End with exactly two lines and nothing else:\n'
        'Answer: X\nConfidence: Y (X in A,B,C,D; Y in [0,1]).\n'
        'Do not output any other text.\n\n'
        f'Question: {question.text}\n\n{opts}\n'
    )

def _ft_extract(raw):
    if not raw: return None
    hits = re.findall(r'ANSWER\s*[:=\-]?\s*([ABCD])\b', raw.upper())
    return hits[-1] if hits else None

# ── Colab / CUDA — transformers + bitsandbytes ────────────────────────────
if _ON_COLAB:
    os.system('pip install -q transformers bitsandbytes accelerate')
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    print(f'[ft] Loading fused model from {_FT_PATH} (4-bit)...')
    _bnb      = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    _ft_hf    = AutoModelForCausalLM.from_pretrained(
                    _FT_PATH, quantization_config=_bnb, device_map='auto',
                    local_files_only=True)
    _ft_tok_hf = AutoTokenizer.from_pretrained(_FT_PATH, local_files_only=True)
    print('[ft] Model ready.')

    def _ft_infer(question):
        prompt = _ft_prompt(question)
        inputs = _ft_tok_hf(prompt, return_tensors='pt',
                            add_special_tokens=False).to(_ft_hf.device)
        t0 = time.time()
        with torch.no_grad():
            out = _ft_hf.generate(
                **inputs, max_new_tokens=_FT_TOKENS,
                do_sample=False, temperature=None, top_p=None,
                pad_token_id=_ft_tok_hf.eos_token_id)
        dt     = time.time() - t0
        raw    = _ft_tok_hf.decode(out[0][inputs['input_ids'].shape[1]:],
                                   skip_special_tokens=True)
        letter = _ft_extract(raw)
        if VERBOSE: print(f'   [ft-raw] {repr(raw[:200])} ({dt:.1f}s)')
        if letter is None:
            if SHOW_QA: print('   [ft] parse failed — random fallback')
            return random.choice(question.options).id
        return question.options[LABELS.index(letter)].id

else:
    print('[finetuned-qwen] This cell requires Google Colab with a GPU runtime.')
    _ft_infer = None

if _ft_infer is not None:
    run_strategy('finetuned-qwen', _ft_infer, comp_ids=[3])

Extracting model from Drive zip...


---
## Final comparison

Accuracy and response time for every strategy on the same 47 fixed Maths questions.  
The dashed line in the right chart marks the **30-second game limit**.


In [ ]:
plot_comparison('Maths')


## Save results to disk

In [ ]:
import os, json as _json
from datetime import datetime

_TS  = datetime.now().strftime('%Y%m%d_%H%M')
_DIR = 'plots'
os.makedirs(_DIR, exist_ok=True)

_path = f'results_{_TS}.json'
with open(_path, 'w') as _f:
    _json.dump({'timestamp': _TS, 'results': RESULTS}, _f, indent=2)
print(f'Results saved -> {_path}')

# Save comparison plot
import matplotlib
_old = matplotlib.rcParams.get('savefig.dpi', 100)
plot_comparison('Maths')
print(f'Done.')


---
## Live game — validate best strategy

Run the best strategy on the **real live game** (5 rounds, Maths category).  
Edit `BEST` to force a specific strategy, or leave it as `None` to auto-pick from RESULTS.


In [ ]:
import numpy as np

BEST = 'hybrid-math-llama'   # set to None to auto-pick from RESULTS

if BEST is None:
    BEST = max(
        (n for n in RESULTS if 'Maths' in RESULTS[n]),
        key=lambda n: np.mean([g['accuracy'] for g in RESULTS[n]['Maths']])
    )
print(f'Running strategy: {BEST}')

use_live_game()    # switch from offline benchmark to the live game
s = STRATEGIES[BEST]
run_strategy(f'{BEST} (live)', s['fn'], comp_ids=[3],
             after_answer=s['after_answer'], before_game=s['before_game'])
plot_strategy(f'{BEST} (live)')
